In [24]:
import numpy as np
#import time
import config.config as config
from validation import Validation
from Environment import Env
from Agent_LSTM import *
from pathlib import Path

import sys; sys.path.append('../analysis/')
from my_utils import reset_seeds

In [27]:
def training(datapath, seed_number, Actor, Critic, task='gain', 
             init_expnoise_std=0.8, TOTAL_EPISODE=1.5e5, get_self_action=True, pro_noise=0.2, obs_noise=0.1):
    # get configures
    if task == 'gain':
        arg = config.ConfigGain(datapath, get_self_action=get_self_action)
    elif task == 'gain_control':
        arg = config.ConfigGainControl(datapath, pro_noise=pro_noise, obs_noise=obs_noise)
    elif 'noise' in task:
        if '_' in task:
            arg = config.ConfigNoiseControl(datapath, task=task.split('_')[1])
        else:
            arg = config.ConfigNoise(datapath)
    else:
        raise ValueError('No such a task!')
            
    arg.SEED_NUMBER = seed_number
    arg.save()
    
    # reproducibility
    reset_seeds(arg.SEED_NUMBER)

    # initialize environment and agent
    env = Env(arg)
    agent = Agent(arg, Actor, Critic)
    validator = Validation(arg.task, agent_name='LSTM')
    
    # define exploration noise
    init_expnoise_std = init_expnoise_std
    noise = ActionNoise(arg.ACTION_DIM, mean=0, std=init_expnoise_std)

    # Remove observation noise in the beginning to help learning in the early stage.
    agent.bstep.obs_noise_range = None
    
    # Loop now
    tot_t = 0
    episode = agent.initial_episode
    reward_log = []
    rewarded_trial_log = []
    step_log = []
    actor_loss_log = 0
    critic_loss_log = 0
    dist_log = []

    LOG_FREQ = 100
    VALIDATION_FREQ = 500
    decrease_lr = True
    REPLAY_PERIOD = 4
    PRE_LEARN_PERIOD = arg.BATCH_SIZE * 50

    enable_mirror_traj = True
    pre_phase=True

    TOTAL_EPISODE = TOTAL_EPISODE


    # Start loop
    while episode < TOTAL_EPISODE:
        # initialize a trial
        cross_start_threshold = False
        reward = torch.zeros(1, 1, 1)

        x = env.reset()
        agent.bstep.reset(env.pro_gains)
        last_action = torch.zeros(1, 1, arg.ACTION_DIM)
        last_action_raw = last_action.clone()

        state = torch.cat([x[-arg.OBS_DIM:].view(1, 1, -1), last_action,
                           env.target_position_obs.view(1, 1, -1)], dim=2).to(arg.device)

        hiddenin = None

        states = []
        actions = []
        rewards = []

        for t in range(arg.EPISODE_LEN):
            # 1. Check start threshold.
            if not cross_start_threshold and (last_action_raw.abs() > arg.TERMINAL_ACTION).any():
                cross_start_threshold = True

            # 2. Take an action based on current state 
            # and previous hidden & cell states of LSTM units.
            action, action_raw, hiddenout = agent.select_action(state, hiddenin, action_noise=noise)

            # 3. Track next x in the environment.
            next_x, reached_target, relative_dist = env(x, action, t)

            # 4. Next observation given next x.
            next_ox = agent.bstep(next_x)
            next_state = torch.cat([next_ox.view(1, 1, -1), action,
                                    env.target_position_obs.view(1, 1, -1)], dim=2).to(arg.device)

            # 5. Check whether stop.
            is_stop = env.is_stop(x, action)

            # 6. Give reward if stopped.          
            if is_stop and cross_start_threshold:
                reward = env.return_reward(x, reward_mode='mixed')

            # 7. Append data.
            states.append(state)
            actions.append(action)
            rewards.append(reward)

            # 8. Update timestep.
            last_action_raw = action_raw
            last_action = action
            state = next_state
            x = next_x
            hiddenin = hiddenout
            tot_t += 1

            # 9. Update model.
            if len(agent.memory.memory) > PRE_LEARN_PERIOD and tot_t % REPLAY_PERIOD == 0:
                #start_time = time.time()
                actor_loss, critic_loss = agent.learn()
                actor_loss_log += actor_loss
                critic_loss_log += critic_loss
                #time_all.append(time.time() - start_time)

            # 10. whether break.
            if is_stop and cross_start_threshold:
                break


        # End of one trial, store trajectory into buffer.
        states = torch.cat(states)
        actions = torch.cat(actions).to(arg.device)
        rewards = torch.cat(rewards).to(arg.device)
        agent.memory.push(states, actions, rewards) 

        if enable_mirror_traj and noise.std != init_expnoise_std:
            # store mirrored trajectories reflected along y-axis
            agent.memory.push(*agent.mirror_traj(states, actions), rewards) 

        #log stuff
        reward_log.append(reward.item())
        rewarded_trial_log.append(int(reached_target & is_stop))
        step_log.append(t + 1)
        dist_log.append(relative_dist.item())

        if episode % LOG_FREQ == LOG_FREQ - 1:
            print(f"t: {tot_t}, Ep: {episode}, action std: {noise.std:0.2f}")
            print(f"mean steps: {np.mean(step_log):0.3f}, "
                  f"mean reward: {np.mean(reward_log):0.3f}, "
                  f"rewarded fraction: {np.mean(rewarded_trial_log):0.3f}, "
                  f"relative distance: {np.mean(dist_log) * arg.LINEAR_SCALE:0.3f}, "
                  f"obs noise: {agent.bstep.obs_noise_range}")

            if decrease_lr and (validator.data.reward_fraction > 0.95).any():
                noise.reset(mean=0, std=0.4)
                agent.actor_optim.param_groups[0]['lr'] = arg.decayed_lr
                agent.critic_optim.param_groups[0]['lr'] = arg.decayed_lr
                decrease_lr = False
                print('Noise and learning rate are changed.')

            if (arg.freeze_belief or arg.freeze_policy) and (validator.data.reward_fraction == 1).any():
                assert not all([arg.freeze_belief, arg.freeze_policy])
                grad_bool = False if arg.freeze_belief else True
                for name, param in agent.actor.named_parameters():
                    param.requires_grad = grad_bool if 'rnn' in name and 'l0' in name else not grad_bool
                arg.freeze_policy = arg.freeze_belief = False
                print('Part of the actor is freezed.')

            if noise.std == init_expnoise_std and np.mean(rewarded_trial_log) > 0.2:
                noise.reset(mean=0, std=0.5)
                agent.bstep.obs_noise_range = arg.obs_noise_range
                agent.memory.reset()
                tot_t = 0
                episode = 0
                pre_phase = False

            reward_log = []
            rewarded_trial_log = []
            step_log = []
            actor_loss_log = 0
            critic_loss_log = 0
            dist_log = []

        # saving and validation
        if episode % VALIDATION_FREQ == VALIDATION_FREQ - 1:
            # save
            agent.save(save_memory=False, episode=episode, pre_phase=pre_phase, full_param=False)

            if noise.std < init_expnoise_std and decrease_lr:
            #if noise.std < init_expnoise_std:
                validator(agent, episode).to_csv(arg.data_path / f'{arg.filename}.csv', index=False)
                agent.bstep.obs_noise_range = arg.obs_noise_range

        episode += 1

In [28]:
actors = ['Actor1']
critics = ['Critic3']
tasks = ['gain']
seeds = [[30]]
init_expnoise_std = 0.8
TOTAL_EPISODE = 1e5
get_self_action = True
pro_noise = 0.3
obs_noise = 0

In [ ]:
for actor, critic, task, seed_ in zip(actors, critics, tasks, seeds):
    for seed in seed_:
        if task == 'gain_control':
            datapath = Path.cwd().parents[1] / f'agents' / f'{actor}{critic}' / task \
                        / f'{pro_noise}_{obs_noise}' / f'seed{seed}'
        else:
            datapath = Path.cwd().parents[1] / f'agents' / f'{actor}{critic}' / task / f'seed{seed}'
        exec(f'from {actor} import *'); exec(f'from {critic} import *')
        training(datapath, seed, Actor, Critic, task, init_expnoise_std, TOTAL_EPISODE, 
                 get_self_action=get_self_action, pro_noise=pro_noise, obs_noise=obs_noise)